In [ ]:
def main(datasources, start_date, end_date):
    """
    ML微结构因子 v12：Transformer + LGB L1 融合版。
    复刻 0.97 方案核心架构：PatchTST (权重 0.15) + L1 Enhanced (权重 0.85)。

    Transformer:
      - 预训练权重从同目录 transformer_model_seed7_raw19.json 加载
      - 输入: 19 原始特征 (OHLCV + 3档bid/ask) × 240 个 5min bar
      - PatchTST: patch_size=12, d_model=256, 4 layers, 8 heads
      - 训练: 2019-2022 bar5m, IC loss, 20 epochs, seed=7
      - 推理: CPU forward pass, 确定性 (权重为常量)

    LGB L1:
      - 与 v8f 完全相同: 83 日频特征, regression_l1, 700 trees
      - 训练: 2019-2022 bar1m→5m→日频, 固定种子确定性

    融合: final = 0.15 * transformer_zscore + 0.85 * lgb_zscore
    """
    import gc
    import json as _json
    import os as _os
    import numpy as np
    import pandas as pd
    import dai
    import lightgbm as lgb
    import torch
    import torch.nn as nn

    np.random.seed(43)
    torch.manual_seed(43)

    # ================================================================
    # 配置
    # ================================================================
    TRAIN_START = '2019-01-01 00:00:00'
    TRAIN_END = '2022-12-31 23:59:59'
    TRAIN_TABLE = 'bigalpha_2026_stock_bar1m'
    WARMUP_DAYS = 45
    CHUNK_SIZE = 120
    MIN_BARS = 20
    T_WEIGHT = 0.15   # Transformer 权重
    L_WEIGHT = 0.85   # LGB 权重

    bar1m = datasources["bar1m"]

    # ================================================================
    # Transformer 模型定义（与训练代码一致）
    # ================================================================
    class PatchEmbedding(nn.Module):
        def __init__(self, n_feat, patch_size, d_model):
            super().__init__()
            self.patch_size = patch_size
            self.proj = nn.Linear(n_feat * patch_size, d_model)
        def forward(self, x):
            B, L, C = x.shape
            n_patch = L // self.patch_size
            x = x[:, :n_patch * self.patch_size, :].reshape(B, n_patch, self.patch_size * C)
            return self.proj(x)

    class StockTransformer(nn.Module):
        def __init__(self, n_feat=19, patch_size=12, d_model=256, nhead=8,
                     nlayers=4, dim_ff=512, dropout=0.1, seq_len=240):
            super().__init__()
            n_patch = seq_len // patch_size
            self.patch_embed = PatchEmbedding(n_feat, patch_size, d_model)
            self.pos = nn.Parameter(torch.zeros(1, n_patch, d_model))
            layer = nn.TransformerEncoderLayer(
                d_model, nhead, dim_ff, dropout,
                batch_first=True, activation="gelu", norm_first=True,
            )
            self.encoder = nn.TransformerEncoder(layer, nlayers, enable_nested_tensor=False)
            self.head = nn.Sequential(
                nn.LayerNorm(d_model), nn.Dropout(dropout), nn.Linear(d_model, 1),
            )
        def forward(self, x):
            h = self.patch_embed(x) + self.pos
            h = self.encoder(h).mean(dim=1)
            return self.head(h).squeeze(-1)

    def _load_transformer():
        """从同目录加载 Transformer 权重。"""
        model_name = "transformer_model_seed7_raw19.json"
        # 搜索多个可能的路径
        search_dirs = [
            _os.getcwd(),
            _os.path.dirname(_os.path.abspath("__file__")),
            "/home/bigquant",
            ".",
        ]
        # 也搜索 cwd 下的子目录（平台可能把文件放在子目录）
        cwd = _os.getcwd()
        for item in _os.listdir(cwd):
            p = _os.path.join(cwd, item)
            if _os.path.isdir(p):
                search_dirs.append(p)

        model_path = None
        for d in search_dirs:
            candidate = _os.path.join(d, model_name)
            if _os.path.exists(candidate):
                model_path = candidate
                break

        if model_path is None:
            # 尝试找 part 文件合并
            for d in search_dirs:
                if not _os.path.isdir(d):
                    continue
                parts = sorted([f for f in _os.listdir(d)
                               if f.startswith(model_name + ".part")])
                if parts:
                    model_path = _os.path.join(d, model_name)
                    with open(model_path, "wb") as out_f:
                        for p in parts:
                            with open(_os.path.join(d, p), "rb") as pf:
                                out_f.write(pf.read())
                    break

        if model_path is None:
            # 最后尝试 find
            import subprocess
            try:
                result_find = subprocess.run(
                    ["find", "/", "-name", model_name, "-type", "f"],
                    capture_output=True, text=True, timeout=10
                )
                lines = [l.strip() for l in result_find.stdout.strip().split("\n") if l.strip()]
                if lines:
                    model_path = lines[0]
            except Exception:
                pass

        if model_path is None:
            raise FileNotFoundError(
                f"找不到 {model_name}，搜索了: {search_dirs}，"
                f"cwd={cwd}, cwd_contents={_os.listdir(cwd)}"
            )

        with open(model_path, "r") as f:
            payload = _json.load(f)

        cfg = payload["model_cfg"]
        model = StockTransformer(
            n_feat=cfg["n_feat"], patch_size=cfg["patch_size"],
            d_model=cfg["d_model"], nhead=cfg["nhead"],
            nlayers=cfg["nlayers"], dim_ff=cfg["dim_ff"],
            dropout=0.0,  # 推理关闭 dropout
            seq_len=cfg["seq_len"],
        )
        sd = {}
        for k, meta in payload["state_dict"].items():
            sd[k] = torch.tensor(meta["data"],
                                 dtype=getattr(torch, meta["dtype"])).reshape(meta["shape"])
        model.load_state_dict(sd)
        model.eval()

        mean = np.array(payload["mean"], dtype=np.float32)
        std = np.array(payload["std"], dtype=np.float32)
        feat_cols = payload["feature_cols"]
        seq_len = payload["seq_len"]
        del payload, sd
        gc.collect()
        return model, mean, std, feat_cols, seq_len

    # ================================================================
    # Transformer 推理
    # ================================================================
    LOG1P_COLS_T = ["volume", "amount", "deal_number",
                    "ask_volume1", "ask_volume2", "ask_volume3",
                    "bid_volume1", "bid_volume2", "bid_volume3"]

    def _transformer_predict(bar5m_df, t_model, t_mean, t_std, t_feat_cols, t_seq_len,
                              output_start, output_end):
        """Transformer 推理: 5m bar → 240-bar 窗口 → 预测。"""
        bar5m_df = bar5m_df.sort_values(["instrument", "date"]).reset_index(drop=True)
        for c in LOG1P_COLS_T:
            if c in bar5m_df.columns:
                bar5m_df[c] = np.log1p(bar5m_df[c].clip(lower=0).astype("float32"))
        for c in t_feat_cols:
            if c not in bar5m_df.columns:
                bar5m_df[c] = 0.0
            else:
                bar5m_df[c] = bar5m_df[c].fillna(0.0).astype("float32")

        out_start = pd.Timestamp(output_start).normalize()
        out_end = pd.Timestamp(output_end).normalize()

        wins, keys = [], []
        for ins, sub in bar5m_df.groupby("instrument", sort=False):
            if len(sub) < t_seq_len:
                continue
            sub = sub.sort_values("date").reset_index(drop=True)
            feats = sub[t_feat_cols].to_numpy(np.float32)
            day_arr = sub["date"].dt.normalize().to_numpy()
            last_bar = np.flatnonzero(np.append(day_arr[1:] != day_arr[:-1], True))
            dates_by_day = day_arr[last_bar]
            for k, p in enumerate(last_bar):
                d = pd.Timestamp(dates_by_day[k])
                if p + 1 < t_seq_len or d < out_start or d > out_end:
                    continue
                wins.append(feats[p - t_seq_len + 1: p + 1])
                keys.append((d, ins))

        if not keys:
            return pd.DataFrame(columns=["date", "instrument", "pred_t"])

        X = np.stack(wins, axis=0)
        X = (X - t_mean) / t_std
        del wins
        gc.collect()

        preds = []
        with torch.no_grad():
            for i in range(0, len(X), 2048):
                batch = torch.from_numpy(X[i:i+2048])
                preds.append(t_model(batch).numpy())
        preds = np.concatenate(preds, axis=0).astype(np.float32)

        result = pd.DataFrame(keys, columns=["date", "instrument"])
        result["pred_t"] = preds
        return result

    # ================================================================
    # 5m bar SQL (与 v8f 完全相同)
    # ================================================================
    BAR_COLS = """
                open, high, low, close, volume, amount, deal_number,
                ask_price1, bid_price1,
                ask_volume1, ask_volume2, ask_volume3,
                bid_volume1, bid_volume2, bid_volume3
    """

    # Transformer 需要额外的列
    T_EXTRA_COLS = "ask_price2, ask_price3, bid_price2, bid_price3"

    def _bar_cte(table_name, query_start, query_end, native_5m):
        """LGB 日频聚合用的 5m CTE（与 v8f 一致）。"""
        where = f"""
            WHERE date >= TIMESTAMP '{query_start}'
              AND date <= TIMESTAMP '{query_end}'
              AND close > 0
              AND ask_price1 > 0 AND bid_price1 > 0 AND ask_price1 >= bid_price1
        """
        if native_5m:
            return f"""
        cte_bar AS (
            SELECT date, instrument, {BAR_COLS}
            FROM {table_name}
            {where}
        )"""
        return f"""
        cte_bar AS (
            SELECT
                time_bucket(INTERVAL 5 MINUTE, date - INTERVAL 1 MINUTE)
                    + INTERVAL 5 MINUTE AS date,
                instrument,
                ARG_MIN(open, date)  AS open,
                MAX(high)            AS high,
                MIN(low)             AS low,
                ARG_MAX(close, date) AS close,
                SUM(volume)          AS volume,
                SUM(amount)          AS amount,
                SUM(deal_number)     AS deal_number,
                ARG_MAX(ask_price1, date)  AS ask_price1,
                ARG_MAX(bid_price1, date)  AS bid_price1,
                ARG_MAX(ask_volume1, date) AS ask_volume1,
                ARG_MAX(ask_volume2, date) AS ask_volume2,
                ARG_MAX(ask_volume3, date) AS ask_volume3,
                ARG_MAX(bid_volume1, date) AS bid_volume1,
                ARG_MAX(bid_volume2, date) AS bid_volume2,
                ARG_MAX(bid_volume3, date) AS bid_volume3
            FROM {table_name}
            {where}
            GROUP BY 1, 2
        )"""

    def _bar_cte_full(table_name, query_start, query_end):
        """Transformer 用的 5m CTE（含 3 档 ask/bid price）。"""
        where = f"""
            WHERE date >= TIMESTAMP '{query_start}'
              AND date <= TIMESTAMP '{query_end}'
              AND close > 0
              AND ask_price1 > 0 AND bid_price1 > 0 AND ask_price1 >= bid_price1
        """
        return f"""
        cte_bar AS (
            SELECT
                time_bucket(INTERVAL 5 MINUTE, date - INTERVAL 1 MINUTE)
                    + INTERVAL 5 MINUTE AS date,
                instrument,
                ARG_MIN(open, date)  AS open,
                MAX(high)            AS high,
                MIN(low)             AS low,
                ARG_MAX(close, date) AS close,
                SUM(volume)          AS volume,
                SUM(amount)          AS amount,
                SUM(deal_number)     AS deal_number,
                ARG_MAX(ask_price1, date)  AS ask_price1,
                ARG_MAX(ask_price2, date)  AS ask_price2,
                ARG_MAX(ask_price3, date)  AS ask_price3,
                ARG_MAX(bid_price1, date)  AS bid_price1,
                ARG_MAX(bid_price2, date)  AS bid_price2,
                ARG_MAX(bid_price3, date)  AS bid_price3,
                ARG_MAX(ask_volume1, date) AS ask_volume1,
                ARG_MAX(ask_volume2, date) AS ask_volume2,
                ARG_MAX(ask_volume3, date) AS ask_volume3,
                ARG_MAX(bid_volume1, date) AS bid_volume1,
                ARG_MAX(bid_volume2, date) AS bid_volume2,
                ARG_MAX(bid_volume3, date) AS bid_volume3
            FROM {table_name}
            {where}
            GROUP BY 1, 2
        )"""

    def _fetch_5m_raw(table_name, query_start, query_end, inst_list):
        """取 5m bar 的全部列（Transformer 用）。"""
        if not inst_list:
            return pd.DataFrame()
        sql = f"""
        WITH{_bar_cte_full(table_name, query_start, query_end)}
        SELECT date, instrument, open, high, low, close, volume, amount, deal_number,
               ask_price1, ask_price2, ask_price3,
               bid_price1, bid_price2, bid_price3,
               ask_volume1, ask_volume2, ask_volume3,
               bid_volume1, bid_volume2, bid_volume3
        FROM cte_bar
        """
        df = dai.query(sql, filters={"date": [query_start, query_end], "instrument": inst_list},
                       compression=True).df()
        if df.empty:
            return df
        df["date"] = pd.to_datetime(df["date"])
        df["instrument"] = df["instrument"].astype(str)
        for c in df.columns:
            if c not in ("date", "instrument"):
                df[c] = pd.to_numeric(df[c], errors="coerce").astype("float32")
        return df


    # ================================================================
    # LGB 日频特征聚合（与 v8f 完全相同）
    # ================================================================
    def _fetch_daily(table_name, query_start, query_end, inst_list, native_5m):
        if not inst_list:
            return pd.DataFrame()

        sql = f"""
        WITH{_bar_cte(table_name, query_start, query_end, native_5m)},
        cte_base AS (
            SELECT
                date,
                instrument,
                strftime(date, '%Y-%m-%d') AS trading_day,
                strftime(date, '%H%M')     AS hm,
                open, high, low, close, volume, amount, deal_number,
                CASE WHEN volume > 0 THEN amount / volume END AS bar_vwap,
                CASE WHEN close > 0
                     THEN (ask_price1 - bid_price1) / close END AS bar_spread,
                CASE WHEN (COALESCE(bid_volume1,0) + COALESCE(ask_volume1,0)) > 0
                     THEN (COALESCE(bid_volume1,0) - COALESCE(ask_volume1,0))
                        / (COALESCE(bid_volume1,0) + COALESCE(ask_volume1,0)) END AS bar_imb1,
                CASE WHEN (COALESCE(bid_volume1,0)+COALESCE(bid_volume2,0)+COALESCE(bid_volume3,0)
                          +COALESCE(ask_volume1,0)+COALESCE(ask_volume2,0)+COALESCE(ask_volume3,0)) > 0
                     THEN (COALESCE(bid_volume1,0)+COALESCE(bid_volume2,0)+COALESCE(bid_volume3,0)
                          -COALESCE(ask_volume1,0)-COALESCE(ask_volume2,0)-COALESCE(ask_volume3,0))
                        / (COALESCE(bid_volume1,0)+COALESCE(bid_volume2,0)+COALESCE(bid_volume3,0)
                          +COALESCE(ask_volume1,0)+COALESCE(ask_volume2,0)+COALESCE(ask_volume3,0)) END AS bar_depth_imb3
            FROM cte_bar
        ),
        cte_ret AS (
            SELECT
                *,
                lag(close, 1) OVER (PARTITION BY instrument, trading_day ORDER BY date) AS prev_close
            FROM cte_base
        ),
        cte_daily AS (
            SELECT
                trading_day,
                instrument,
                COUNT(*) AS n_bars,

                -- 全日 OHLCV
                ARG_MIN(open,  date) AS open_first,
                ARG_MAX(close, date) AS close_last,
                MAX(high) AS high_max,
                MIN(low)  AS low_min,
                SUM(volume) AS volume_sum,
                SUM(amount) AS amount_sum,
                SUM(deal_number) AS deal_sum,
                nanstd(close) AS close_std,
                AVG(bar_vwap) AS vwap_bar_mean,

                -- 成交集中度（HHI 风格，日内分钟成交量平方和 / 总量平方）
                SUM(CAST(volume AS DOUBLE) * CAST(volume AS DOUBLE)) AS vol_sq_sum,

                -- 分钟收益分布
                AVG(CASE WHEN prev_close > 0 THEN close / prev_close - 1.0 END) AS bar_ret_mean,
                nanstd(CASE WHEN prev_close > 0 THEN close / prev_close - 1.0 END) AS bar_ret_std,

                -- 微结构：全日
                AVG(bar_spread) AS spread_mean,
                nanstd(bar_spread) AS spread_std,
                ARG_MAX(bar_spread, date) AS spread_last,
                ARG_MIN(bar_spread, date) AS spread_first,
                AVG(bar_imb1) AS imb1_mean,
                nanstd(bar_imb1) AS imb1_std,
                ARG_MAX(bar_imb1, date) AS imb1_last,
                ARG_MIN(bar_imb1, date) AS imb1_first,
                AVG(bar_depth_imb3) AS depth_imb3_mean,
                ARG_MAX(bar_depth_imb3, date) AS depth_imb3_last,

                -- 时段价格锚点
                ARG_MAX(CASE WHEN hm <= '1000' THEN close END,
                        CASE WHEN hm <= '1000' THEN date  END) AS close_first30,
                ARG_MAX(CASE WHEN hm <= '1130' THEN close END,
                        CASE WHEN hm <= '1130' THEN date  END) AS close_am,
                ARG_MIN(CASE WHEN hm >= '1300' THEN open  END,
                        CASE WHEN hm >= '1300' THEN date  END) AS open_pm,
                ARG_MIN(CASE WHEN hm >= '1430' THEN open  END,
                        CASE WHEN hm >= '1430' THEN date  END) AS open_last30,

                -- 时段量能
                SUM(CASE WHEN hm <= '1000' THEN volume ELSE 0 END) AS volume_first30,
                SUM(CASE WHEN hm <= '1130' THEN volume ELSE 0 END) AS volume_am,
                SUM(CASE WHEN hm >= '1300' THEN volume ELSE 0 END) AS volume_pm,
                SUM(CASE WHEN hm >= '1430' THEN volume ELSE 0 END) AS volume_last30,
                SUM(CASE WHEN hm <= '1000' THEN amount ELSE 0 END) AS amount_first30,
                SUM(CASE WHEN hm >= '1430' THEN amount ELSE 0 END) AS amount_last30,
                SUM(CASE WHEN hm <= '1000' THEN deal_number ELSE 0 END) AS deal_first30,
                SUM(CASE WHEN hm >= '1430' THEN deal_number ELSE 0 END) AS deal_last30,

                -- 时段 VWAP / 微结构
                AVG(CASE WHEN hm <= '1000' THEN bar_vwap END) AS vwap_first30_mean,
                AVG(CASE WHEN hm >= '1430' THEN bar_vwap END) AS vwap_last30_mean,
                AVG(CASE WHEN hm <= '1000' THEN bar_spread END) AS spread_first30_mean,
                AVG(CASE WHEN hm >= '1430' THEN bar_spread END) AS spread_last30_mean,
                AVG(CASE WHEN hm <= '1000' THEN bar_imb1 END) AS imb1_first30_mean,
                AVG(CASE WHEN hm >= '1430' THEN bar_imb1 END) AS imb1_last30_mean,
                AVG(CASE WHEN hm <= '1000' THEN bar_depth_imb3 END) AS depth_imb3_first30_mean,
                AVG(CASE WHEN hm >= '1430' THEN bar_depth_imb3 END) AS depth_imb3_last30_mean
            FROM cte_ret
            GROUP BY trading_day, instrument
        )
        SELECT
            CAST(trading_day AS DATETIME) AS date,
            instrument,
            n_bars,
            open_first, close_last, high_max, low_min,
            volume_sum, amount_sum, deal_sum, close_std, vwap_bar_mean, vol_sq_sum,
            bar_ret_mean, bar_ret_std,
            spread_mean, spread_std, spread_last, spread_first,
            imb1_mean, imb1_std, imb1_last, imb1_first,
            depth_imb3_mean, depth_imb3_last,
            close_first30, close_am, open_pm, open_last30,
            volume_first30, volume_am, volume_pm, volume_last30,
            amount_first30, amount_last30, deal_first30, deal_last30,
            vwap_first30_mean, vwap_last30_mean,
            spread_first30_mean, spread_last30_mean,
            imb1_first30_mean, imb1_last30_mean,
            depth_imb3_first30_mean, depth_imb3_last30_mean
        FROM cte_daily
        WHERE n_bars >= {MIN_BARS}
        """
        df = dai.query(
            sql,
            filters={"date": [query_start, query_end], "instrument": inst_list},
            compression=True,
        ).df()

        if df.empty:
            return df

        df["date"] = pd.to_datetime(df["date"])
        df["instrument"] = df["instrument"].astype(str)
        df = df.drop_duplicates(subset=["date", "instrument"], keep="first")
        df = df.sort_values(["instrument", "date"]).reset_index(drop=True)
        for c in df.columns:
            if c not in ("date", "instrument"):
                df[c] = pd.to_numeric(df[c], errors="coerce").astype("float32")
        return df

    # ============================================================
    # pandas 侧派生特征（对齐 0.97 报告的 add_enhanced_lag_features）
    # ============================================================
    def _safe_div(a, b):
        a = np.asarray(a, dtype="float64")
        b = np.asarray(b, dtype="float64")
        return a / np.where(np.abs(b) > 1e-12, b, np.nan)

    def _add_features(df):
        if df.empty:
            return df
        df = df.copy()

        close = df["close_last"].to_numpy("float64")
        open_ = df["open_first"].to_numpy("float64")
        high = df["high_max"].to_numpy("float64")
        low = df["low_min"].to_numpy("float64")
        vol = df["volume_sum"].to_numpy("float64")
        amt = df["amount_sum"].to_numpy("float64")
        deal = df["deal_sum"].to_numpy("float64")
        vwap_day = _safe_div(amt, vol)

        # --- 日内形态 ---
        df["ret_oc"] = _safe_div(close, open_) - 1.0
        df["range_hl"] = _safe_div(high, low) - 1.0
        df["close_pos"] = _safe_div(close - low, high - low)
        df["close_to_high"] = _safe_div(close, high) - 1.0
        df["close_to_low"] = _safe_div(close, low) - 1.0
        df["close_vwap_dev"] = _safe_div(close, vwap_day) - 1.0
        df["close_bar_vwap_dev"] = _safe_div(close, df["vwap_bar_mean"].to_numpy("float64")) - 1.0
        df["close_std_rel"] = _safe_div(df["close_std"].to_numpy("float64"), close)

        # --- 量能规模 ---
        df["log_volume"] = np.log1p(np.clip(vol, 0, None))
        df["log_amount"] = np.log1p(np.clip(amt, 0, None))
        df["log_deal"] = np.log1p(np.clip(deal, 0, None))
        df["avg_trade_amount"] = np.log1p(_safe_div(amt, deal))
        df["avg_trade_volume"] = np.log1p(_safe_div(vol, deal))
        df["vol_hhi"] = _safe_div(df["vol_sq_sum"].to_numpy("float64"), vol * vol)

        # --- 时段收益 ---
        df["ret_first30"] = _safe_div(df["close_first30"], df["open_first"]) - 1.0
        df["ret_am"] = _safe_div(df["close_am"], df["open_first"]) - 1.0
        df["ret_pm"] = _safe_div(df["close_last"], df["open_pm"]) - 1.0
        df["ret_last30"] = _safe_div(df["close_last"], df["open_last30"]) - 1.0
        df["ret_lunch_gap"] = _safe_div(df["open_pm"], df["close_am"]) - 1.0
        df["ret_am_to_close"] = _safe_div(df["close_last"], df["close_am"]) - 1.0
        df["close_vwap_first30_dev"] = _safe_div(df["close_first30"], df["vwap_first30_mean"]) - 1.0
        df["close_vwap_last30_dev"] = _safe_div(df["close_last"], df["vwap_last30_mean"]) - 1.0

        # --- 时段量能占比 ---
        for name in ["first30", "am", "pm", "last30"]:
            df[f"vol_{name}_ratio"] = _safe_div(df[f"volume_{name}"], df["volume_sum"])
        df["amount_first30_ratio"] = _safe_div(df["amount_first30"], df["amount_sum"])
        df["amount_last30_ratio"] = _safe_div(df["amount_last30"], df["amount_sum"])
        df["deal_first30_ratio"] = _safe_div(df["deal_first30"], df["deal_sum"])
        df["deal_last30_ratio"] = _safe_div(df["deal_last30"], df["deal_sum"])
        df["avg_trade_amount_first30"] = np.log1p(_safe_div(df["amount_first30"], df["deal_first30"]))
        df["avg_trade_amount_last30"] = np.log1p(_safe_div(df["amount_last30"], df["deal_last30"]))

        # --- 微结构日内漂移（尾盘 vs 全日）---
        df["spread_last_vs_mean"] = df["spread_last"] - df["spread_mean"]
        df["spread_last30_vs_mean"] = df["spread_last30_mean"] - df["spread_mean"]
        df["spread_first30_vs_mean"] = df["spread_first30_mean"] - df["spread_mean"]
        df["imb1_last_vs_mean"] = df["imb1_last"] - df["imb1_mean"]
        df["imb1_last30_vs_mean"] = df["imb1_last30_mean"] - df["imb1_mean"]
        df["imb1_first30_vs_mean"] = df["imb1_first30_mean"] - df["imb1_mean"]
        df["depth_imb3_last_vs_mean"] = df["depth_imb3_last"] - df["depth_imb3_mean"]
        df["depth_imb3_last30_vs_mean"] = df["depth_imb3_last30_mean"] - df["depth_imb3_mean"]

        # --- 跨日时序 ---
        g = df.groupby("instrument", sort=False)
        df["ret_1"] = _safe_div(close, g["close_last"].shift(1).to_numpy("float64")) - 1.0
        for w in [2, 3, 5, 10, 20]:
            df[f"ret_{w}"] = _safe_div(close, g["close_last"].shift(w).to_numpy("float64")) - 1.0
        for w in [5, 10, 20]:
            minp = max(2, w // 2)
            df[f"ret1_mean_{w}"] = g["ret_1"].transform(lambda s: s.rolling(w, min_periods=minp).mean())
            df[f"ret1_std_{w}"] = g["ret_1"].transform(lambda s: s.rolling(w, min_periods=minp).std())
            df[f"range_mean_{w}"] = g["range_hl"].transform(lambda s: s.rolling(w, min_periods=minp).mean())
            df[f"logvol_z_{w}"] = df["log_volume"] - g["log_volume"].transform(
                lambda s: s.rolling(w, min_periods=minp).mean())
        for c in ["ret_last30", "vol_last30_ratio", "spread_last30_mean", "imb1_last30_mean"]:
            df[f"{c}_mean_5"] = g[c].transform(lambda s: s.rolling(5, min_periods=2).mean())
            df[f"{c}_std_5"] = g[c].transform(lambda s: s.rolling(5, min_periods=2).std())

        # --- 标签 ---
        df["fwd_ret_1"] = _safe_div(g["close_last"].shift(-1).to_numpy("float64"), close) - 1.0

        df = df.replace([np.inf, -np.inf], np.nan)
        return df

    # 需要在全截面计算的 rank 特征（分批时不能算，留到 concat 后统一算）
    RANK_COLS = [
        "ret_1", "ret_5", "ret_oc", "range_hl", "close_pos",
        "ret_first30", "ret_am", "ret_pm", "ret_last30",
        "vol_first30_ratio", "vol_last30_ratio", "log_volume",
        "close_vwap_dev", "spread_last", "spread_last30_mean",
        "imb1_last", "imb1_last30_mean", "depth_imb3_last",
        "ret1_std_20", "logvol_z_20", "vol_hhi",
    ]

    def _add_cs_ranks(df):
        for c in RANK_COLS:
            df[f"cs_rank_{c}"] = df.groupby("date")[c].rank(pct=True) - 0.5
        return df

    DROP_COLS = {"date", "instrument", "fwd_ret_1", "target_z"}

    # ============================================================
    # 分批取数 + 特征构建
    # ============================================================
    def _chunks(seq, size):
        for i in range(0, len(seq), size):
            yield seq[i:i + size]

    def build_panel(table_name, query_start, query_end, output_start, inst_list, native_5m):
        parts = []
        out_start_dt = pd.to_datetime(output_start)
        out_end_dt = pd.to_datetime(query_end)
        for chunk in _chunks(inst_list, CHUNK_SIZE):
            raw = _fetch_daily(table_name, query_start, query_end, chunk, native_5m)
            if raw.empty:
                del raw
                gc.collect()
                continue
            feat = _add_features(raw)
            keep = feat[(feat["date"] >= out_start_dt) & (feat["date"] <= out_end_dt)]
            if not keep.empty:
                parts.append(keep.copy())
            del raw, feat, keep
            gc.collect()
        if not parts:
            return pd.DataFrame()
        panel = pd.concat(parts, ignore_index=True)
        del parts
        gc.collect()
        return _add_cs_ranks(panel)


    # 全历史成分股的取数区间：写死的常量，不随 start_date / end_date 变化。
    # dai 不允许查 bigalpha_2026_instruments 时不带 filters（PermissionException），
    # 所以这里用一个足够宽的固定区间来代替「不加过滤」。
    UNIVERSE_START = '2019-01-01 00:00:00'
    UNIVERSE_END = '2030-12-31 23:59:59'

    def _universe(sd=None, ed=None):
        """取成分股列表。sd/ed 为 None 时用写死的全历史区间。"""
        if sd is None or ed is None:
            sd, ed = UNIVERSE_START, UNIVERSE_END
        d = dai.query(
            "SELECT DISTINCT instrument FROM bigalpha_2026_instruments",
            filters={"date": [sd, ed]},
        ).df()
        out = sorted(d["instrument"].astype(str).tolist())
        del d
        gc.collect()
        return out

    # 训练股票池：按写死的训练区间取，与评估窗口无关，
    # 也避免用评估期成分股去筛训练样本（那会把测试期指数成分信息带进训练集）。
    train_universe = _universe(TRAIN_START, TRAIN_END)

    # 预测股票池：全历史成分股，不带日期过滤。
    # 必须与 [start_date, end_date] 解耦 —— 否则 end_date 变化会改变截面成员，
    # 使 cs_rank_* 特征在同一交易日算出不同值，被未来函数自检判为差异。
    pred_universe = _universe()

    # ============================================================
    # 第1步：训练（写死区间 + 写死训练表）
    # ============================================================
    # 不加 try/except：取数失败必须直接抛出，避免静默回退导致本地与线上模型不一致
    train_panel = build_panel(TRAIN_TABLE, TRAIN_START, TRAIN_END, TRAIN_START,
                              train_universe, False)
    if train_panel.empty:
        raise RuntimeError("训练面板为空：检查 TRAIN_TABLE 与训练区间")

    train_panel = train_panel.dropna(subset=["fwd_ret_1"])
    # target = 截面 zscore，clip[-5,5]（0.97 方案的 target_z）
    train_panel["target_z"] = train_panel.groupby("date")["fwd_ret_1"].transform(
        lambda s: (s - s.mean()) / (s.std() + 1e-12)
    )
    train_panel = train_panel.dropna(subset=["target_z"])

    feature_cols = [c for c in train_panel.columns if c not in DROP_COLS]
    X_train = train_panel[feature_cols].fillna(0.0).to_numpy("float32")
    y_train = train_panel["target_z"].clip(-5, 5).to_numpy("float32")
    del train_panel
    gc.collect()

    # 0.97 方案 L1 Enhanced 超参（报告 4.2）+ 完整确定性开关
    # 确定性必须显式打开：平台的未来函数自检会用 selftest / ahead 两张表
    # 各跑一次 main() 并逐格比对（容差 rtol=1e-5, atol=1e-8）。
    # 每次调用都会重新训练，若 LightGBM 非确定性，两次模型不同 -> 全部日期
    # 预测值不一致 -> 被误判为「疑似未来函数」。
    # 对齐 0.97 仓库 reproducibility.py:114 lgb_repro_params() 的做法。
    LGB_SEED = 43
    model = lgb.LGBMRegressor(
        objective="regression_l1",
        n_estimators=700,
        learning_rate=0.025,
        num_leaves=63,
        min_child_samples=100,
        subsample=0.85,
        subsample_freq=1,
        colsample_bytree=0.85,
        reg_alpha=0.1,
        reg_lambda=2.0,
        random_state=LGB_SEED,
        seed=LGB_SEED,
        bagging_seed=LGB_SEED,
        feature_fraction_seed=LGB_SEED,
        data_random_seed=LGB_SEED,
        drop_seed=LGB_SEED,
        deterministic=True,
        force_col_wise=True,
        n_jobs=4,
        verbose=-1,
    )
    model.fit(X_train, y_train)
    del X_train, y_train
    gc.collect()



    # ============================================================
    # 第2步：Transformer 推理
    # ============================================================
    t_model, t_mean, t_std, t_feat_cols, t_seq_len = _load_transformer()

    start_dt = pd.to_datetime(start_date)
    pred_query_start = (start_dt - pd.Timedelta(days=WARMUP_DAYS)).strftime("%Y-%m-%d %H:%M:%S")

    # 取 5m bar raw 数据（Transformer 需要逐 bar）
    bar5m_raw = _fetch_5m_raw(bar1m, pred_query_start, end_date, pred_universe)
    if not bar5m_raw.empty:
        t_preds = _transformer_predict(bar5m_raw, t_model, t_mean, t_std, t_feat_cols, t_seq_len,
                                        start_date, end_date)
    else:
        t_preds = pd.DataFrame(columns=["date", "instrument", "pred_t"])
    del bar5m_raw, t_model
    gc.collect()

    # ============================================================
    # 第3步：LGB 预测
    # ============================================================
    test_panel = build_panel(bar1m, pred_query_start, end_date, start_date, pred_universe, False)
    if test_panel.empty:
        raise RuntimeError("预测面板为空：检查 datasources['bar1m'] 与评估区间")

    X_test = test_panel[feature_cols].fillna(0.0).to_numpy("float32")
    result = test_panel[["date", "instrument"]].copy()
    result["pred_l"] = model.predict(X_test)
    del test_panel, X_test, model
    gc.collect()

    # ============================================================
    # 第4步：融合 Transformer + LGB
    # ============================================================
    # 各自逐日 zscore
    result["pred_l"] = result.groupby("date")["pred_l"].transform(
        lambda s: (s - s.mean()) / (s.std() + 1e-8)
    )

    if not t_preds.empty:
        t_preds["date"] = pd.to_datetime(t_preds["date"]).dt.normalize()
        t_preds["pred_t"] = t_preds.groupby("date")["pred_t"].transform(
            lambda s: (s - s.mean()) / (s.std() + 1e-8)
        )
        result["date_norm"] = pd.to_datetime(result["date"]).dt.normalize()
        result = pd.merge(result, t_preds[["date", "instrument", "pred_t"]],
                          left_on=["date_norm", "instrument"],
                          right_on=["date", "instrument"],
                          how="left", suffixes=("", "_t"))
        result.drop(columns=["date_t", "date_norm"], inplace=True, errors="ignore")
        result["pred_t"] = result["pred_t"].fillna(0.0)
        result["factor"] = L_WEIGHT * result["pred_l"] + T_WEIGHT * result["pred_t"]
        result.drop(columns=["pred_l", "pred_t"], inplace=True)
    else:
        result["factor"] = result["pred_l"]
        result.drop(columns=["pred_l"], inplace=True)
    del t_preds
    gc.collect()

    # 最终截面 zscore + clip
    result["factor"] = result.groupby("date")["factor"].transform(
        lambda s: (s - s.mean()) / (s.std() + 1e-8)
    ).clip(-5.0, 5.0)

    # ============================================================
    # 第5步：对齐股票池
    # ============================================================
    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={"date": [start_date, end_date]},
    ).df()
    stk_pool["date"] = pd.to_datetime(stk_pool["date"])
    stk_pool["instrument"] = stk_pool["instrument"].astype(str)

    result = pd.merge(result, stk_pool, how="right", on=["date", "instrument"])
    del stk_pool
    gc.collect()

    result["factor"] = result["factor"].replace([np.inf, -np.inf], np.nan)
    result["factor"] = result.groupby("date", sort=False)["factor"].transform(
        lambda s: s.fillna(s.median())
    )
    result = result.dropna(subset=["factor"]).reset_index(drop=True)
    return result[["date", "instrument", "factor"]]

